In [20]:
import pandas as pd
import numpy as np

# ==========================================
# 1. 配置文件路径 (请替换为实际路径)
# ==========================================
ED_STAYS_PATH = '../data/edstays.csv.gz'
CXR_META_PATH = '../data/mimic-cxr-2.0.0-metadata.csv.gz'
CXR_LABEL_PATH = '../data/mimic-cxr-2.0.0-chexpert.csv.gz'
MASTER_DATA_PATH = '../data/mimic_all.csv'
OUTPUT_PATH = '../data/final_ed_cxr_master_cohort.csv'

print("--- Step 1: 加载基础数据 ---")
ed_df = pd.read_csv(ED_STAYS_PATH, usecols=['subject_id', 'stay_id', 'intime', 'outtime'])
cxr_meta = pd.read_csv(CXR_META_PATH, usecols=['subject_id', 'study_id', 'dicom_id', 'StudyDate', 'ViewPosition'])
cxr_labels = pd.read_csv(CXR_LABEL_PATH, usecols=['subject_id', 'study_id', 'Pneumonia'])
master_df = pd.read_csv(MASTER_DATA_PATH)

print(f"初始急诊就诊总数: {len(ed_df)}")
print(f"初始X光图像总数: {len(cxr_meta)}")

# ==========================================
# Step 2: X光片基础过滤 (仅正位 + 有明确肺炎标签)
# ==========================================
print("\n--- Step 2: X光片视角与标签过滤 ---")
# 合并元数据和标签
cxr_full = pd.merge(cxr_meta, cxr_labels, on=['subject_id', 'study_id'], how='inner')

# 过滤1：保留所有正位片 (AP 和 PA)
cxr_full = cxr_full[cxr_full['ViewPosition'].isin(['AP', 'PA'])]

# 过滤2：仅保留 Pneumonia 明确为 1.0 或 0.0 的记录 (丢弃 NaN 和 -1)
cxr_full = cxr_full[cxr_full['Pneumonia'].isin([1.0, 0.0])]
print(f"过滤掉侧位和标签不明确的片子后，剩余候选 X光片: {len(cxr_full)} 张")

# ==========================================
# Step 3: 时间对齐 (核心修复步骤！)
# 先把X光片的时间和急诊时间拼在一起，强制要求 X光片必须在急诊期间拍摄
# ==========================================
print("\n--- Step 3: 执行严格时间对齐 ---")
# 将 X 光的 StudyDate (YYYYMMDD) 转为 datetime
cxr_full['cxr_date'] = pd.to_datetime(cxr_full['StudyDate'].astype(str), format='%Y%m%d', errors='coerce')

# 将 ED 的入科和出科转为 datetime
ed_df['intime'] = pd.to_datetime(ed_df['intime'])
ed_df['outtime'] = pd.to_datetime(ed_df['outtime'])

# 基于 subject_id 合并 (产生所有急诊记录与该患者所有X光的组合)
aligned_df = pd.merge(ed_df, cxr_full, on='subject_id', how='inner')

# 在急诊窗口内
aligned_df = aligned_df[
    (aligned_df['cxr_date'] >= aligned_df['intime']) & 
    (aligned_df['cxr_date'] <= aligned_df['outtime'])
]
print(f"成功找到发生在【急诊窗口内】的 X光片关联记录: {len(aligned_df)} 条")

# ==========================================
# Step 4: 提取符合条件的“第一次”就诊记录
# ==========================================
print("\n--- Step 4: 提取单患单次就诊 ---")
# 按患者ID 和 急诊入科时间排序 (确保是最早的一次急诊)
aligned_df = aligned_df.sort_values(by=['subject_id', 'intime'])

# 针对同一个患者，我们只保留他人生中第一次发生“急诊+拍正位片+有标签”的这一套记录
final_cohort = aligned_df.drop_duplicates(subset=['subject_id'], keep='first')
print(f"去重后，最终确认独立患者人数: {len(final_cohort)} 人")

# ==========================================
# Step 5: 与 Master Data 融合获取临床特征
# ==========================================
print("\n--- Step 5: 提取临床特征 (Master Data) ---")
# 注意：我们这里必须通过 subject_id 和 stay_id 双键匹配！
# 这样才能保证提取的特征绝对是拍片子那次急诊的特征。

# 为了防泄漏，建议手动指定你要保留的 Master Data 列，这里用所有列演示
final_master = pd.merge(
    final_cohort[['subject_id', 'stay_id', 'study_id', 'dicom_id', 'Pneumonia', 'cxr_date', 'StudyDate', 'ViewPosition']], 
    master_df, 
    on=['subject_id', 'stay_id'], 
    how='inner'
)

print(f"最终生成！完美匹配结构化特征的队列人数: {len(final_master)} 人")

# 检查正负样本比例
pos_n = len(final_master[final_master['Pneumonia'] == 1.0])
neg_n = len(final_master[final_master['Pneumonia'] == 0.0])
print(f"最终数据集中: 肺炎阳性 {pos_n} 例, 阴性 {neg_n} 例")

# 保存最终干净的 Master 表
#final_master.to_csv(OUTPUT_PATH, index=False)
#print(f"\n所有清洗完成。输出文件: {OUTPUT_PATH}")

--- Step 1: 加载基础数据 ---
初始急诊就诊总数: 425087
初始X光图像总数: 377110

--- Step 2: X光片视角与标签过滤 ---
过滤掉侧位和标签不明确的片子后，剩余候选 X光片: 41525 张

--- Step 3: 执行严格时间对齐 ---
成功找到发生在【急诊窗口内】的 X光片关联记录: 1838 条

--- Step 4: 提取单患单次就诊 ---
去重后，最终确认独立患者人数: 1603 人

--- Step 5: 提取临床特征 (Master Data) ---
最终生成！完美匹配结构化特征的队列人数: 1601 人
最终数据集中: 肺炎阳性 525 例, 阴性 1076 例


In [21]:
import pandas as pd

# 假设 final_master 是你上一步运行完的结果 (1601行)
# 我们定义最终要保留的“肺炎研究专用列”

# 1. Image Identifiers (X-Ray related)
image_cols = ['subject_id', 'stay_id', 'study_id', 'dicom_id', 'StudyDate', 'ViewPosition']

# 2. Target Label
label_col = ['Pneumonia']

# 3. Clinical Features (Structured Input)
# We select columns highly relevant to pneumonia diagnosis
clinical_cols = [
    # Demographics
    'age', 'gender', 
    # Vital Signs at Triage (Earliest point of care)
    'triage_temperature', 'triage_heartrate', 'triage_resprate', 
    'triage_o2sat', 'triage_sbp', 'triage_dbp', 'triage_acuity',
    # Respiratory Symptoms (Chief Complaints)
    'chiefcom_shortness_of_breath', 'chiefcom_cough', 'chiefcom_fever_chills',
    # Relevant Comorbidities (To distinguish from Heart Failure/COPD)
    'cci_Pulmonary', 'cci_CHF', 'score_CCI',
    # pneumonia related
    'outcome_bac_pne', 'outcome_viral_pne', 'outcome_all_pne'
]

# Create the final dataframe
final_cohort_study = final_master[image_cols + label_col + clinical_cols].copy()

# Simple Pre-processing: Fill missing vital signs with median to ensure usability
for col in ['triage_temperature', 'triage_heartrate', 'triage_resprate', 'triage_o2sat']:
    final_cohort_study[col] = final_cohort_study[col].fillna(final_cohort_study[col].median())

print(f"Final Cohort Shape: {final_cohort_study.shape}")
print(f"Columns selected: {final_cohort_study.columns.tolist()}")

# Save as the ultimate master file for your model
#final_cohort_study.to_csv('../data/final_pneumonia_multimodal_cohort.csv', index=False)

Final Cohort Shape: (1601, 25)
Columns selected: ['subject_id', 'stay_id', 'study_id', 'dicom_id', 'StudyDate', 'ViewPosition', 'Pneumonia', 'age', 'gender', 'triage_temperature', 'triage_heartrate', 'triage_resprate', 'triage_o2sat', 'triage_sbp', 'triage_dbp', 'triage_acuity', 'chiefcom_shortness_of_breath', 'chiefcom_cough', 'chiefcom_fever_chills', 'cci_Pulmonary', 'cci_CHF', 'score_CCI', 'outcome_bac_pne', 'outcome_viral_pne', 'outcome_all_pne']


In [22]:
import pandas as pd

# 1. Load the cohort we just saved
df = final_cohort_study.copy()

# 2. Apply the logic for Pneumonia == 0
# If X-ray says no pneumonia, we ensure all clinical outcome pneumonia flags are 0
mask_neg = (df['Pneumonia'] == 0)
df.loc[mask_neg, ['outcome_bac_pne', 'outcome_viral_pne', 'outcome_all_pne']] = 0

# 3. Apply the logic for Pneumonia == 1
# If X-ray says pneumonia, we ensure at least the 'all_pne' flag is 1
mask_pos = (df['Pneumonia'] == 1)
df.loc[mask_pos, 'outcome_all_pne'] = 1

# 4. Calculate the positivity rates (Percentage of 1s)
cols_to_stats = ['Pneumonia', 'outcome_bac_pne', 'outcome_viral_pne', 'outcome_all_pne']
stats = df[cols_to_stats].mean() * 100

print("--- Final Positivity Rates (%) ---")
for col, rate in stats.items():
    print(f"{col}: {rate:.2f}%")

# 5. Count the absolute numbers for clarity
print("\n--- Absolute Counts (Positive Cases) ---")
print(df[cols_to_stats].sum())


--- Final Positivity Rates (%) ---
Pneumonia: 32.79%
outcome_bac_pne: 1.87%
outcome_viral_pne: 1.00%
outcome_all_pne: 32.79%

--- Absolute Counts (Positive Cases) ---
Pneumonia            525.0
outcome_bac_pne       30.0
outcome_viral_pne     16.0
outcome_all_pne      525.0
dtype: float64


In [23]:
# 1. Convert Boolean-like columns (True/False or strings) to 1/0
# This handles the chief complaint columns you specified
cols_to_fix = ['chiefcom_shortness_of_breath', 'chiefcom_cough', 'chiefcom_fever_chills']
for col in cols_to_fix:
    # We use .astype(int) which maps True->1, False->0
    # If they are strings 'True'/'False', we map them explicitly
    if df[col].dtype == 'object':
        df[col] = df[col].map({'True': 1, 'False': 0, True: 1, False: 0}).fillna(0).astype(int)
    else:
        df[col] = df[col].astype(int)

# 2. Gender Encoding: Majority group as 1, others as 0
# First, let's find out which gender is the majority
gender_counts = df['gender'].value_counts()
majority_gender = gender_counts.idxmax()
print(f"Majority gender is '{majority_gender}' with {gender_counts.max()} cases.")

# Apply the encoding: 1 for majority, 0 for minority
df['gender'] = (df['gender'] == majority_gender).astype(int)
print(f"Gender encoded: {majority_gender} -> 1, Others -> 0")

# 3. Check data types of all selected columns
# Let's assume you have your 24 columns in the final selection
# If you haven't defined the list, we'll take the current columns
print("\n--- Data Types of the Final Columns ---")
print(df.dtypes)

# 4. Final verification: Check for any remaining non-numeric columns
# Excluding identifiers like subject_id and dicom_id
print("\n--- Summary of Non-Numeric Columns (if any) ---")
print(df.select_dtypes(exclude=['number']).columns.tolist())


Majority gender is 'F' with 834 cases.
Gender encoded: F -> 1, Others -> 0

--- Data Types of the Final Columns ---
subject_id                        int64
stay_id                           int64
study_id                          int64
dicom_id                         object
StudyDate                         int64
ViewPosition                     object
Pneumonia                       float64
age                               int64
gender                            int64
triage_temperature              float64
triage_heartrate                float64
triage_resprate                 float64
triage_o2sat                    float64
triage_sbp                      float64
triage_dbp                      float64
triage_acuity                   float64
chiefcom_shortness_of_breath      int64
chiefcom_cough                    int64
chiefcom_fever_chills             int64
cci_Pulmonary                     int64
cci_CHF                           int64
score_CCI                         int64
outc

In [24]:
import pandas as pd

# 1. 设置显示最大列数（None 表示不限制，显示全部）
pd.set_option('display.max_columns', None)

# 2. 建议同时设置显示宽度，防止换行错乱
pd.set_option('display.width', 1000)

# 3. 如果你想看前几行的所有列数据
display(df.head())
print(f"shape: {df.shape}")


,subject_id,stay_id,study_id,dicom_id,StudyDate,ViewPosition,Pneumonia,age,gender,triage_temperature,triage_heartrate,triage_resprate,triage_o2sat,triage_sbp,triage_dbp,triage_acuity,chiefcom_shortness_of_breath,chiefcom_cough,chiefcom_fever_chills,cci_Pulmonary,cci_CHF,score_CCI,outcome_bac_pne,outcome_viral_pne,outcome_all_pne
0,10003019,34451930,50543252,3f4a324f-7967a6b4-91edf0c8-94fbefd4-32402065,21751101,PA,1.0,73,0,36.611111,84.0,18.0,100.0,132.0,70.0,2.0,0,0,0,1,0,7,0,0,1
1,10003956,36506816,53245562,dc29d33e-bcf77ecf-c4fca6b6-8ea2ed29-d71aee14,21721221,PA,0.0,31,0,36.500000,95.0,18.0,100.0,146.0,98.0,3.0,0,0,0,0,0,0,0,0,0
2,10008816,33521511,51161169,b339fcfb-9786a1fa-811f53ab-020b753c-5adae68a,21290521,PA,0.0,40,0,36.388889,101.0,14.0,96.0,123.0,76.0,2.0,0,0,0,0,0,0,0,0,0
3,10015860,37977734,54943790,049385a6-012ffc8e-96dcf51b-0e58326e-42e859c6,21890523,AP,1.0,56,0,37.944444,140.0,20.0,95.0,139.0,70.0,1.0,0,0,0,0,0,6,0,0,1
4,10046282,33844394,59720796,9c5c9f70-5499fd6f-b612606d-ab74edeb-dd3cafcc,21791027,PA,0.0,68,0,36.111111,88.0,16.0,100.0,188.0,74.0,2.0,0,0,0,0,0,2,0,0,0


shape: (1601, 25)


In [25]:
df.to_csv('../data/mimic_ed_cxr_pneumonia_multimodal_cohort.csv', index=False)

In [26]:
import pandas as pd

# Load your multimodal cohort
df = pd.read_csv('../data/mimic_ed_cxr_pneumonia_multimodal_cohort.csv')

# Double check the count
print(f"Preparing download list for {len(df)} images.")

# Construct PhysioNet download URLs
def build_url(row):
    subject_id = str(row['subject_id'])
    prefix = 'p' + subject_id[:2]
    study_id = 's' + str(row['study_id'])
    dicom_id = row['dicom_id']
    return f"https://physionet.org/files/mimic-cxr-jpg/2.1.0/files/{prefix}/p{subject_id}/{study_id}/{dicom_id}.jpg"

df['url'] = df.apply(build_url, axis=1)

# Save to url_list.txt
df['url'].to_csv('../data/url_list_final.txt', index=False, header=False)
print("Download list 'url_list_final.txt' is ready in your data folder.")

Preparing download list for 1601 images.
Download list 'url_list_final.txt' is ready in your data folder.


In [27]:
import pandas as pd
import os

# ==========================================
# 1. 配置文件路径
# ==========================================
# 你的最终 1601 人名单
COHORT_PATH = '../data/mimic_ed_cxr_pneumonia_multimodal_cohort.csv'
# 存放图片的根目录
IMAGE_ROOT = '../data/files/'

print("Starting cleanup process...")

# 2. 获取“黄金名单”中的所有合法文件名 (dicom_id)
df = pd.read_csv(COHORT_PATH)
# 加上 .jpg 后缀，方便匹配
valid_images = set(df['dicom_id'].astype(str) + '.jpg')
print(f"Target count: {len(valid_images)} images to keep.")

# 3. 遍历本地文件夹并删除不在名单中的文件
deleted_count = 0
kept_count = 0

# os.walk 会递归遍历 files 文件夹下的所有子文件夹
for root, dirs, files in os.walk(IMAGE_ROOT):
    for filename in files:
        if filename.endswith('.jpg'):
            if filename in valid_images:
                kept_count += 1
            else:
                # 执行删除操作
                file_path = os.path.join(root, filename)
                os.remove(file_path)
                deleted_count += 1

print(f"\nCleanup Complete!")
print(f"Images kept: {kept_count}")
print(f"Redundant images deleted: {deleted_count}")

# 4. 可选：清理空的子文件夹
print("Cleaning up empty directories...")
for root, dirs, files in os.walk(IMAGE_ROOT, topdown=False):
    for name in dirs:
        dir_path = os.path.join(root, name)
        if not os.listdir(dir_path): # 如果文件夹为空
            os.rmdir(dir_path)

Starting cleanup process...
Target count: 1601 images to keep.

Cleanup Complete!
Images kept: 20
Redundant images deleted: 160
Cleaning up empty directories...
